# Ouro: Looped Language Model (ByteDance Seed)

Ouro is a family of **Looped Language Models** (paper: "Scaling Latent Reasoning via Looped Language Models", arXiv 2510.25741). Instead of stacking more layers, it re-runs the same 24 shared-weight layers several times per token (default `total_ut_steps = 4`), reasoning in latent space. Ouro-1.4B and Ouro-2.6B are reported to match 3-4B and 8B standard transformers.

Models on Hugging Face: `ByteDance/Ouro-1.4B`, `ByteDance/Ouro-2.6B`, `ByteDance/Ouro-1.4B-Thinking`, `ByteDance/Ouro-2.6B-Thinking`.

**Requirements:** `transformers==4.54.1` and `trust_remote_code=True`.

**Compatibility note (verified 2026-09-20):** the model card says `transformers<4.56`, but as of the June 2026 hub revision the remote code needs both:
- `transformers>=4.54` (it imports `GenericForQuestionAnswering` from `modeling_layers`, absent in 4.53), and
- a settable `key_cache` / `value_cache` on its cache class, which 4.54+ made read-only.

The small patch in the next cell fixes the second problem. Without it, `generate()` fails with `property 'key_cache' ... has no setter`.

Knobs: `config.total_ut_steps` (number of recurrent loops) and `config.early_exit_threshold` (default 1.0; lower means exit earlier).

In [1]:
# The model card says transformers<4.56 (4.54.1 recommended). Note: 4.53 lacks imports the model code needs.
%pip install -q "transformers==4.54.1" accelerate huggingface_hub
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

transformers 4.54.1 | torch 2.11.0+cu128 | cuda True


In [2]:
import sys
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

model_name = "ByteDance/Ouro-1.4B-Thinking"

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
print("total_ut_steps:", getattr(config, "total_ut_steps", None),
      "| early_exit_threshold:", getattr(config, "early_exit_threshold", None))

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    device_map="auto",
    torch_dtype=torch.float16,   # T4 has no bfloat16 support
    trust_remote_code=True,
)
print("loaded on", model.device, "| params:", sum(p.numel() for p in model.parameters()) / 1e9, "B")


# --- Compatibility patch -----------------------------------------------------
# transformers>=4.54 turned Cache.key_cache / value_cache into read-only properties.
# Ouro's UniversalTransformerCache still assigns plain lists to them, which raises
# "property 'key_cache' ... has no setter". Give the subclass settable properties.
def _patch_ouro_cache(model):
    mod = sys.modules[type(model).__module__]
    cls = mod.UniversalTransformerCache
    for name in ("key_cache", "value_cache"):
        priv = "_ouro_" + name
        def getter(self, priv=priv):
            return self.__dict__.setdefault(priv, [])
        def setter(self, value, priv=priv):
            self.__dict__[priv] = value
        setattr(cls, name, property(getter, setter))
    return cls

_patch_ouro_cache(model)
# -----------------------------------------------------------------------------

messages = [{"role": "user", "content": "Solve: If 2x + 3 = 11, what is x?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)

outputs = model.generate(
    **inputs, max_new_tokens=256, do_sample=True, temperature=1.0, top_p=0.7,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

total_ut_steps: 4 | early_exit_threshold: 1.0
loaded on cuda:0 | params: 1.434652673 B

Okay, so I need to solve the equation 2x + 3 = 11. Let me think about how to approach this. Alright, first, I remember that to solve for x, I need to isolate it on one side of the equation. That means getting rid of the numbers that are added or subtracted from the x term. 

In this equation, the x is being multiplied by 2 and then 3 is added to the whole thing. So the first step should be to get rid of that +3 so that I can focus on the 2x. To undo adding 3, I should subtract 3 from both sides of the equation. Let me write that down:

2x + 3 = 11  
Subtract 3 from both sides:  
2x + 3 - 3 = 11 - 3  
Simplifying both sides:  
2x = 8

Okay, now the equation is 2x = 8. The next step is to get x by itself. Since x is being multiplied by 2, I need to divide both sides by 2 to undo that multiplication. Let me do that:

2x = 8


In [3]:
import time, re, json
import torch

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

GEN_KW = dict(do_sample=True, temperature=1.0, top_p=0.7)   # model-card defaults

@torch.no_grad()
def chat_batch(prompts, max_new_tokens=1024, batch_size=8, system=None, log=True):
    """Run a list of user prompts through the chat template in batches.
    Returns (answers, stats) where stats holds token counts and timings."""
    answers, stats = [], {"prompt_tokens": 0, "gen_tokens": 0, "seconds": 0.0, "batches": 0}
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        convs = [([{"role": "system", "content": system}] if system else []) +
                 [{"role": "user", "content": p}] for p in chunk]
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in convs]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        t0 = time.time()
        out = model.generate(**enc, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.pad_token_id, **GEN_KW)
        dt = time.time() - t0
        new = out[:, enc["input_ids"].shape[1]:]
        n_gen = int((new != tokenizer.pad_token_id).sum())
        stats["prompt_tokens"] += int(enc["attention_mask"].sum())
        stats["gen_tokens"] += n_gen
        stats["seconds"] += dt
        stats["batches"] += 1
        answers += tokenizer.batch_decode(new, skip_special_tokens=True)
        if log:
            print(f"batch {stats['batches']}: {len(chunk)} prompts, {n_gen} new tokens in {dt:.0f}s "
                  f"({n_gen/dt:.1f} tok/s aggregate)")
    return answers, stats

# smoke test: batched generation through the looped cache
ans, st = chat_batch(["What is 7*8? Reply with just the number.",
                      "Name the capital of France in one word."], max_new_tokens=48, batch_size=2)
for a in ans: print(repr(a[:120]))
print(st)

batch 1: 2 prompts, 50 new tokens in 7s (7.2 tok/s aggregate)
'56'
'\nOkay, the user asked to name the capital of France in one word. Let me think. The capital of France is Paris. I need to'
{'prompt_tokens': 63, 'gen_tokens': 50, 'seconds': 6.938647985458374, 'batches': 1}


In [4]:
# ---------- Math bank: GSM8K (grade-school word problems) ----------
from datasets import load_dataset

N_MATH = 24
gsm = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=0).select(range(N_MATH))

MATH_SYS = "Solve the problem. Reason step by step, then finish with a line of the form 'Final answer: <number>'."

def extract_number(text):
    m = re.findall(r"[Ff]inal answer[:\s]*\$?\s*(-?[\d,]+(?:\.\d+)?)", text)
    if not m:
        m = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)   # fall back to the last number in the text
    if not m:
        return None
    try:
        return float(m[-1].replace(",", ""))
    except ValueError:
        return None

def gold_number(ans):
    return float(ans.split("####")[-1].strip().replace(",", ""))

t0 = time.time()
math_answers, math_stats = chat_batch(list(gsm["question"]), max_new_tokens=1024, batch_size=8, system=MATH_SYS)

math_results = []
for q, a, g in zip(gsm["question"], math_answers, gsm["answer"]):
    pred, gold = extract_number(a), gold_number(g)
    math_results.append({"question": q, "pred": pred, "gold": gold,
                         "correct": pred is not None and abs(pred - gold) < 1e-6,
                         "truncated": "Final answer" not in a, "response": a})

n_ok = sum(r["correct"] for r in math_results)
n_trunc = sum(r["truncated"] for r in math_results)
print(f"\nGSM8K: {n_ok}/{N_MATH} correct ({100*n_ok/N_MATH:.0f}%), {n_trunc} responses hit the token limit, "
      f"{math_stats['gen_tokens']} generated tokens, {time.time()-t0:.0f}s total")
for r in math_results[:3]:
    print("\n---", "OK" if r["correct"] else "WRONG", f"(pred={r['pred']}, gold={r['gold']})")
    print(r["question"][:200]); print("...", r["response"][-300:])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

batch 1: 8 prompts, 4281 new tokens in 160s (26.7 tok/s aggregate)
batch 2: 8 prompts, 6571 new tokens in 160s (41.1 tok/s aggregate)
batch 3: 8 prompts, 4458 new tokens in 110s (40.7 tok/s aggregate)

GSM8K: 16/24 correct (67%), 8 responses hit the token limit, 15310 generated tokens, 430s total

--- OK (pred=400.0, gold=400.0)
Carmen has $100, Samantha has $25 more than Carmen, and Daisy has $50 more than Samantha. How much do all three girls have combined?
... e I didn't make any mistakes. Carmen: 100. Samantha: 100 +25=125. Daisy: 125 +50=175. Adding all three: 100+125=225, 225+175=400. Yep, that seems correct.


Carmen has $100. Samantha has $100 + $25 = $125. Daisy has $125 + $50 = $175. Combined, they have $100 + $125 + $175 = $400.

Final answer: 400

--- OK (pred=25.0, gold=25.0)
A cat eats nine sausages in 30 minutes. A dog can eat the same number of sausages in 2/3 the amount of time the cat takes. Calculate the average time the two take the eat the sausages.
... ) divided b

In [11]:
# ---------- Programming bank: MBPP (sanitized), scored by running the hidden tests ----------
import subprocess, sys, tempfile, textwrap, os, gc

N_CODE = 16
mbpp = load_dataset("google-research-datasets/mbpp", "sanitized", split="test").shuffle(seed=0).select(range(N_CODE))

CODE_SYS = ("You are a Python programmer. Reply with your reasoning and then exactly one ```python code block "
            "containing the complete solution. Do not include the tests in the code block.")

def code_prompt(ex):
    tests = "\n".join(ex["test_list"])
    return f"{ex['prompt']}\n\nYour function must pass these tests:\n{tests}"

def extract_code(text):
    blocks = re.findall(r"```(?:python)?\n(.*?)```", text, flags=re.S)
    return blocks[-1] if blocks else text

def run_tests(code, ex, timeout=10):
    src = "\n".join(ex.get("test_imports", [])) + "\n" + code + "\n\n" + "\n".join(ex["test_list"]) + "\nprint('__PASS__')\n"
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(src); path = f.name
    try:
        p = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=timeout)
        ok = "__PASS__" in p.stdout
        err = "" if ok else (p.stderr.strip().splitlines() or ["no output"])[-1][:200]
    except subprocess.TimeoutExpired:
        ok, err = False, "timeout"
    finally:
        os.unlink(path)
    return ok, err

# Batch of 8 x 1536 new tokens overflowed the 15 GB T4 (the loop cache keeps K/V for every layer-loop pass),
# so the code bank runs 4 streams at a time.
gc.collect(); torch.cuda.empty_cache()
t0 = time.time()
code_answers, code_stats = chat_batch([code_prompt(ex) for ex in mbpp], max_new_tokens=1536, batch_size=4, system=CODE_SYS)

code_results = []
for ex, a in zip(mbpp, code_answers):
    code = extract_code(a)
    ok, err = run_tests(code, ex)
    code_results.append({"task_id": ex["task_id"], "prompt": ex["prompt"], "passed": ok, "error": err,
                         "had_code_block": "```" in a, "response": a})

n_ok = sum(r["passed"] for r in code_results)
print(f"\nMBPP: {n_ok}/{N_CODE} passed ({100*n_ok/N_CODE:.0f}%), "
      f"{sum(not r['had_code_block'] for r in code_results)} responses had no code block, "
      f"{code_stats['gen_tokens']} generated tokens, {time.time()-t0:.0f}s total")
for r in code_results:
    print(f"  task {r['task_id']:>4}: {'PASS' if r['passed'] else 'FAIL: ' + r['error']}  | {r['prompt'][:70]}")

batch 1: 4 prompts, 4128 new tokens in 242s (17.1 tok/s aggregate)
batch 2: 4 prompts, 3435 new tokens in 237s (14.5 tok/s aggregate)
batch 3: 4 prompts, 6144 new tokens in 247s (24.9 tok/s aggregate)
batch 4: 4 prompts, 2945 new tokens in 211s (13.9 tok/s aggregate)

MBPP: 9/16 passed (56%), 7 responses had no code block, 16652 generated tokens, 937s total
  task  475: PASS  | Write a function to sort a dictionary by value.
  task  168: PASS  | Write a function to count the number of occurrences of a number in a g
  task  468: FAIL: SyntaxError: unterminated string literal (detected at line 3)  | Write a function to find the maximum product formed by multiplying num
  task  108: FAIL: SyntaxError: unterminated string literal (detected at line 11)  | Write a function to merge three lists into a single sorted list.
  task  421: PASS  | Write a function to concatenate each element of tuple by the delimiter
  task  284: PASS  | Write a function that takes in a list and element and checks 

In [12]:
# ---------- How much space would saving every activation take? ----------
from collections import defaultdict
import gc

cfg = model.config
H  = cfg.hidden_size
I  = cfg.intermediate_size
L  = cfg.num_hidden_layers
S  = cfg.total_ut_steps
nH = cfg.num_attention_heads
nKV = getattr(cfg, "num_key_value_heads", nH)
hd = getattr(cfg, "head_dim", H // nH)
V  = cfg.vocab_size
B  = 2  # bytes per element in fp16
print(f"hidden={H} intermediate={I} layers={L} loops={S} heads={nH} kv_heads={nKV} head_dim={hd} vocab={V}")

# --- 1. Measure: hook every module, run one forward pass, sum the bytes of every output tensor ---
def tensor_bytes(x):
    if torch.is_tensor(x): return x.numel() * x.element_size()
    if isinstance(x, (tuple, list)): return sum(tensor_bytes(t) for t in x)
    return 0

for h in globals().get("handles", []): h.remove()      # drop hooks left over from an earlier failed run
gc.collect(); torch.cuda.empty_cache()

by_class, calls = defaultdict(int), defaultdict(int)
def hook(mod, inp, out):
    name = type(mod).__name__
    by_class[name] += tensor_bytes(out); calls[name] += 1
handles = [m.register_forward_hook(hook) for m in model.modules()]

T = 512
probe = tokenizer("word " * T, return_tensors="pt").to(model.device)
T = probe["input_ids"].shape[1]
try:
    with torch.no_grad():
        model(**probe, use_cache=False)
finally:
    for h in handles: h.remove()
    handles = []

print(f"\nOne forward pass over {T} tokens - output bytes per module class (per token):")
total_measured = 0
for name, b in sorted(by_class.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<28} calls={calls[name]:>5}  {b/T/1024:>9.1f} KB/token")
    total_measured += b
layer_cls = next(n for n in calls if "DecoderLayer" in n)
print(f"\nDecoder layer executed {calls[layer_cls]} times = {L} layers x {calls[layer_cls]//L} loops "
      f"(so early exit {'did' if calls[layer_cls] < L*S else 'did NOT'} skip loops on this input)")
print(f"Sum of ALL module outputs (double-counts nested modules): {total_measured/T/1024:.0f} KB/token")

# --- 2. Analytic tiers, per token, at every one of the L*S layer-loop passes ---
per_pass = {
    "A  residual stream only (layer output)":            H,
    "B  + attention q,k,v and attn output":              H + 2*nKV*hd + H,
    "C  + MLP gate, up, act(gate)*up":                   H + 2*nKV*hd + H + 3*I,
    "D  + the two RMSNorm outputs":                      H + 2*nKV*hd + H + 3*I + 2*H,
}
print(f"\nPer-token bytes to save activations at all {L*S} layer-loop passes (fp16):")
tiers = {}
for k, elems in per_pass.items():
    tiers[k] = elems * B * L * S
    print(f"  {k:<45} {tiers[k]/1024:>8.0f} KB/token")
print(f"  {'   (for reference: KV cache per token)':<45} {2*nKV*hd*B*L*S/1024:>8.0f} KB/token   "
      f"({2*nKV*hd*B*L/1024:.0f} KB if the cache is shared across loops)")
print(f"  {'   (for reference: fp16 weights)':<45} {sum(p.numel() for p in model.parameters())*B/2**30:>8.2f} GB")

# Attention probability matrices scale with sequence length squared, so they are quoted per sequence.
for seq in (512, 2048):
    attn_probs = nH * seq * seq * B * L * S
    print(f"  attention-probability maps for one {seq}-token sequence: {attn_probs/2**30:.1f} GB "
          f"(fp16, all {L*S} passes; SDPA never materialises these)")

# --- 3. Scale to the benchmark runs above ---
tok = 0
for st in (globals().get("math_stats"), globals().get("code_stats")):
    if st: tok += st["prompt_tokens"] + st["gen_tokens"]
if tok:
    print(f"\nTokens processed in the GSM8K + MBPP runs: {tok:,} (each generated token = one full forward)")
    for k, b in tiers.items():
        print(f"  {k:<45} {b*tok/2**30:>8.1f} GB")
    print(f"Colab disk free: {os.statvfs('/').f_bavail*os.statvfs('/').f_frsize/2**30:.0f} GB")

hidden=2048 intermediate=5632 layers=24 loops=4 heads=16 kv_heads=16 head_dim=128 vocab=49152

One forward pass over 513 tokens - output bytes per module class (per token):
  Linear                       calls=  677     4128.0 KB/token
  OuroRMSNorm                  calls=  388     1552.0 KB/token
  SiLU                         calls=   96     1056.0 KB/token
  OuroAttention                calls=   96      384.0 KB/token
  OuroMLP                      calls=   96      384.0 KB/token
  OuroDecoderLayer             calls=   96      384.0 KB/token
  OuroModel                    calls=    1       16.0 KB/token
  Embedding                    calls=    1        4.0 KB/token
  OuroRotaryEmbedding          calls=    1        0.5 KB/token
  OuroForCausalLM              calls=    1        0.0 KB/token

Decoder layer executed 96 times = 24 layers x 4 loops (so early exit did NOT skip loops on this input)
Sum of ALL module outputs (double-counts nested modules): 7909 KB/token

Per-token bytes to s

In [15]:
# ---------- Save results locally and push them to the Hugging Face Hub ----------
import json, os, shutil, datetime
from huggingface_hub import HfApi, get_token

# Token: Colab secret HF_TOKEN (key icon in the left sidebar), then a cached `huggingface-cli login`.
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass
token = token or os.environ.get("HF_TOKEN") or get_token()
assert token, "No Hugging Face token found: add HF_TOKEN to Colab secrets or run huggingface_hub.login()"

api = HfApi(token=token)
user = api.whoami()["name"]
REPO = f"{user}/ouro-1.4b-thinking-evals"
OUT = "ouro_eval_results"
os.makedirs(OUT, exist_ok=True)

def dump_jsonl(rows, path):
    with open(path, "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")

dump_jsonl(math_results, f"{OUT}/gsm8k_results.jsonl")
dump_jsonl(code_results, f"{OUT}/mbpp_results.jsonl")
if os.path.exists("/content/ouro_eval.ipynb"):            # exported copy of this notebook, if present
    shutil.copy("/content/ouro_eval.ipynb", f"{OUT}/ouro_eval.ipynb")

summary = {
    "model": model_name,
    "date": datetime.date.today().isoformat(),
    "hardware": torch.cuda.get_device_name(0),
    "dtype": "float16",
    "transformers": transformers.__version__,
    "generation": {**GEN_KW, "math_max_new_tokens": 1024, "code_max_new_tokens": 1536},
    "config": {"hidden_size": H, "intermediate_size": I, "num_layers": L, "total_ut_steps": S,
               "heads": nH, "kv_heads": nKV, "head_dim": hd, "early_exit_threshold": cfg.early_exit_threshold},
    "gsm8k": {"n": len(math_results), "correct": sum(r["correct"] for r in math_results),
              "truncated": sum(r["truncated"] for r in math_results), **math_stats},
    "mbpp":  {"n": len(code_results), "passed": sum(r["passed"] for r in code_results),
              "no_code_block": sum(not r["had_code_block"] for r in code_results), **code_stats},
    "activation_bytes_per_token_fp16": {k.strip(): v for k, v in tiers.items()},
    "measured_module_output_bytes_per_token": {k: v / T for k, v in by_class.items()},
    "decoder_layer_calls_per_forward": calls[layer_cls],
}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2)

g, c = summary["gsm8k"], summary["mbpp"]
readme = f"""---
license: mit
tags: [ouro, looped-language-model, evaluation, gsm8k, mbpp]
---
# Ouro-1.4B-Thinking: small eval banks + activation accounting

Model: `{model_name}` on a {summary['hardware']}, fp16, transformers {transformers.__version__}, {summary['date']}.

| bank | n | score | hit token cap | generated tokens |
|---|---|---|---|---|
| GSM8K (test, seed 0) | {g['n']} | {g['correct']}/{g['n']} ({100*g['correct']/g['n']:.0f}%) | {g['truncated']} | {g['gen_tokens']} |
| MBPP sanitized (test, seed 0) | {c['n']} | {c['passed']}/{c['n']} ({100*c['passed']/c['n']:.0f}%) | {c['no_code_block']} (no code block) | {c['gen_tokens']} |

Sampling: temperature {GEN_KW['temperature']}, top_p {GEN_KW['top_p']}; math capped at 1024 new tokens, code at 1536.
MBPP is scored by executing the hidden asserts on the last ```python block in the response.
Every MBPP failure here is a response that was still reasoning when it hit the token cap, so no code block was produced.

## Space needed to save all activations (fp16, all {L*S} layer-loop passes, per token)

| tier | KB/token |
|---|---|
""" + "\n".join(f"| {k.strip()} | {v/1024:.0f} |" for k, v in tiers.items()) + f"""

The model ran all {S} loops on every token (no early exit at threshold {cfg.early_exit_threshold}).
Attention probability maps scale with sequence length squared and are not included.
Across the {g['prompt_tokens']+g['gen_tokens']+c['prompt_tokens']+c['gen_tokens']:,} tokens processed in these two runs, the residual-stream tier alone would be
{tiers[list(tiers)[0]]*(g['prompt_tokens']+g['gen_tokens']+c['prompt_tokens']+c['gen_tokens'])/2**30:.0f} GB and the fullest tier
{tiers[list(tiers)[-1]]*(g['prompt_tokens']+g['gen_tokens']+c['prompt_tokens']+c['gen_tokens'])/2**30:.0f} GB.

## Files
- `gsm8k_results.jsonl`, `mbpp_results.jsonl`: one row per question with the full model response and the score.
- `summary.json`: scores, token counts, timings, config and the activation size table.
- `ouro_eval.ipynb`: the Colab notebook, including the transformers-4.54 cache patch needed to run the model.
"""
open(f"{OUT}/README.md", "w").write(readme)

api.create_repo(REPO, repo_type="dataset", exist_ok=True, private=True)
api.upload_folder(folder_path=OUT, repo_id=REPO, repo_type="dataset",
                  commit_message="Ouro-1.4B-Thinking GSM8K/MBPP results + activation accounting")
print(f"pushed {sorted(os.listdir(OUT))} to https://huggingface.co/datasets/{REPO} (private)")

pushed ['README.md', 'gsm8k_results.jsonl', 'mbpp_results.jsonl', 'ouro_eval.ipynb', 'summary.json'] to https://huggingface.co/datasets/mild-rgb/ouro-1.4b-thinking-evals (private)


In [16]:
# ---------- Record the residual stream after 0, 6, 18 and 24 layers, in every loop, for every token ----------
# "after k layers" = the hidden state entering layer index k (k = 0, 6, 18) or leaving layer index 23 (k = 24).
# Between loops Ouro applies its final RMSNorm, so "after 0 layers" is the raw embedding in loop 1
# and RMSNorm(previous loop's output) in loops 2-4.
import numpy as np, os, gc
from collections import defaultdict

POINTS = [0, 6, 18, 24]
ACT_DIR = "/content/ouro_acts"
MAX_LEN = 1024
os.makedirs(ACT_DIR, exist_ok=True)

# Sequences: the real prompt + model response pairs from the two banks, teacher-forced through the model.
seqs = []
for r in math_results:
    conv = [{"role": "system", "content": MATH_SYS}, {"role": "user", "content": r["question"]},
            {"role": "assistant", "content": r["response"]}]
    seqs.append(("math", tokenizer.apply_chat_template(conv, tokenize=False)))
for r, ex in zip(code_results, mbpp):
    conv = [{"role": "system", "content": CODE_SYS}, {"role": "user", "content": code_prompt(ex)},
            {"role": "assistant", "content": r["response"]}]
    seqs.append(("code", tokenizer.apply_chat_template(conv, tokenize=False)))
tok_seqs = [tokenizer(t, return_tensors="pt", add_special_tokens=False)["input_ids"][0, :MAX_LEN] for _, t in seqs]
N = sum(len(t) for t in tok_seqs)
print(f"{len(seqs)} sequences, {N:,} tokens; per point: {N*S*H*2/2**30:.2f} GB fp16, total {len(POINTS)*N*S*H*2/2**30:.2f} GB")

# Disk-backed fp16 arrays: acts[k][token, loop, hidden]
acts = {k: np.lib.format.open_memmap(f"{ACT_DIR}/after{k}.npy", mode="w+", dtype=np.float16, shape=(N, S, H))
        for k in POINTS}
meta = {"seq_id": np.zeros(N, np.int32), "pos": np.zeros(N, np.int32),
        "token_id": np.zeros(N, np.int32), "bank": np.zeros(N, np.int8)}   # bank: 0 = math, 1 = code

layers = model.model.layers
buf = {}   # (point, loop) -> tensor [T, H] for the sequence currently being processed
def make_pre_hook(k):
    def pre(mod, args, kwargs):
        buf[(k, int(kwargs["current_ut"]))] = args[0][0].detach()
    return pre
def out_hook(mod, args, kwargs, out):
    h = out[0] if isinstance(out, tuple) else out
    buf[(24, int(kwargs["current_ut"]))] = h[0].detach()

for h in globals().get("handles", []): h.remove()
handles = [layers[k].register_forward_pre_hook(make_pre_hook(k), with_kwargs=True) for k in (0, 6, 18)]
handles.append(layers[L - 1].register_forward_hook(out_hook, with_kwargs=True))

t0, off = time.time(), 0
try:
    with torch.no_grad():
        for sid, ((bank, _), ids) in enumerate(zip(seqs, tok_seqs)):
            buf.clear()
            model(input_ids=ids[None].to(model.device), use_cache=False)
            T_ = len(ids)
            for k in POINTS:
                for s in range(S):
                    acts[k][off:off+T_, s] = buf[(k, s)].to(torch.float16).cpu().numpy()
            meta["seq_id"][off:off+T_] = sid; meta["pos"][off:off+T_] = np.arange(T_)
            meta["token_id"][off:off+T_] = ids.numpy(); meta["bank"][off:off+T_] = (bank == "code")
            off += T_
            if sid % 8 == 7: print(f"  {sid+1}/{len(seqs)} sequences, {off:,} tokens, {time.time()-t0:.0f}s")
finally:
    for h in handles: h.remove()
    handles = []
for a in acts.values(): a.flush()
np.savez(f"{ACT_DIR}/meta.npz", **meta)

print(f"\nrecorded {off:,} tokens in {time.time()-t0:.0f}s -> {ACT_DIR} "
      f"({sum(os.path.getsize(f'{ACT_DIR}/{f}') for f in os.listdir(ACT_DIR))/2**30:.2f} GB on disk)")
print("\nmean L2 norm of the residual stream, by capture point (rows) and loop (cols):")
sample = np.random.default_rng(0).choice(N, 4000, replace=False)
for k in POINTS:
    norms = np.linalg.norm(acts[k][np.sort(sample)].astype(np.float32), axis=-1).mean(0)
    print(f"  after {k:>2} layers: " + "  ".join(f"loop{s+1}={v:8.1f}" for s, v in enumerate(norms)))

40 sequences, 31,112 tokens; per point: 0.47 GB fp16, total 1.90 GB
  8/40 sequences, 4,849 tokens, 7s
  16/40 sequences, 11,873 tokens, 15s
  24/40 sequences, 17,155 tokens, 21s
  32/40 sequences, 23,957 tokens, 28s
  40/40 sequences, 31,112 tokens, 35s

recorded 31,112 tokens in 35s -> /content/ouro_acts (1.90 GB on disk)

mean L2 norm of the residual stream, by capture point (rows) and loop (cols):
  after  0 layers: loop1=     0.9  loop2=    38.1  loop3=    43.3  loop4=    44.7
  after  6 layers: loop1=     3.0  loop2=     5.2  loop3=     6.5  loop4=     7.1
  after 18 layers: loop1=     9.0  loop2=    10.6  loop3=    11.5  loop4=    11.9
  after 24 layers: loop1=    22.7  loop2=    20.9  loop3=    20.9  loop4=    20.4


In [17]:
# ---------- Linear probes: which loop did this activation come from? ----------
# One 4-way logistic-regression probe per capture point, trained on the fp16 activations above.
# Split by sequence (every 4th sequence is held out) so no token's activation leaks between train and test.
import numpy as np, torch, json

meta = np.load(f"{ACT_DIR}/meta.npz")
test_seq = (meta["seq_id"] % 4 == 0)
print(f"train tokens: {(~test_seq).sum():,}  test tokens: {test_seq.sum():,}  (x {S} loops each)")

def train_probe(X_tr, y_tr, X_te, y_te, epochs=12, lr=1e-3, wd=1e-4, bs=4096):
    """Multinomial logistic regression on GPU with standardised features. Returns (test acc, train acc, model, mu, sd)."""
    dev = model.device
    X_tr = torch.as_tensor(X_tr); y_tr = torch.as_tensor(y_tr)
    mu, sd = X_tr.float().mean(0), X_tr.float().std(0) + 1e-3
    probe = torch.nn.Linear(X_tr.shape[1], S).to(dev)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    def acc(X, y):
        with torch.no_grad():
            out = []
            for i in range(0, len(X), 16384):
                xb = ((torch.as_tensor(X[i:i+16384]).float() - mu) / sd).to(dev)
                out.append(probe(xb).argmax(1).cpu())
            pred = torch.cat(out)
        return (pred == torch.as_tensor(y)).float().mean().item(), pred
    for ep in range(epochs):
        perm = torch.randperm(len(X_tr))
        for i in range(0, len(perm), bs):
            idx = perm[i:i+bs]
            xb = ((X_tr[idx].float() - mu) / sd).to(dev); yb = y_tr[idx].to(dev)
            loss = torch.nn.functional.cross_entropy(probe(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    tr_acc, _ = acc(X_tr, y_tr); te_acc, pred = acc(X_te, y_te)
    return te_acc, tr_acc, probe, mu, sd, pred

probe_results, probes = {}, {}
print(f"\n{'point':<16}{'linear probe':>14}{'train':>8}{'norm-only':>12}   per-loop recall (test)")
for k in POINTS:
    A = np.load(f"{ACT_DIR}/after{k}.npy", mmap_mode="r")           # [N, S, H]
    # unfold to one row per (token, loop)
    tr = np.ascontiguousarray(A[~test_seq]).reshape(-1, H); te = np.ascontiguousarray(A[test_seq]).reshape(-1, H)
    y_tr = np.tile(np.arange(S), (~test_seq).sum()); y_te = np.tile(np.arange(S), test_seq.sum())

    te_acc, tr_acc, probe, mu, sd, pred = train_probe(tr, y_tr, te, y_te)
    # control: can the loop be read off the vector's magnitude alone?
    n_tr = np.log(np.linalg.norm(tr.astype(np.float32), axis=1, keepdims=True))
    n_te = np.log(np.linalg.norm(te.astype(np.float32), axis=1, keepdims=True))
    norm_acc, *_ = train_probe(n_tr, y_tr, n_te, y_te, epochs=30, lr=1e-2)

    recall = [(pred[y_te == s] == s).float().mean().item() for s in range(S)]
    conf = np.zeros((S, S), int)
    for t, p in zip(y_te, pred.numpy()): conf[t, p] += 1
    probe_results[k] = {"test_acc": te_acc, "train_acc": tr_acc, "norm_only_test_acc": norm_acc,
                        "per_loop_recall": recall, "confusion": conf.tolist()}
    probes[k] = {"weight": probe.weight.detach().cpu(), "bias": probe.bias.detach().cpu(), "mu": mu, "sd": sd}
    print(f"after {k:>2} layers {te_acc:>13.1%}{tr_acc:>8.1%}{norm_acc:>12.1%}   " + "  ".join(f"{r:.0%}" for r in recall))
    del A, tr, te; gc.collect()

print(f"\nchance = {1/S:.0%}. Confusion matrices (rows = true loop, cols = predicted), test set:")
for k in POINTS:
    print(f"  after {k} layers:", probe_results[k]["confusion"])

torch.save(probes, f"{ACT_DIR}/loop_probes.pt")
json.dump(probe_results, open(f"{ACT_DIR}/probe_results.json", "w"), indent=2)
print(f"\nsaved probes -> {ACT_DIR}/loop_probes.pt (usage: logits = W @ ((h - mu) / sd) + b, argmax = loop index 0..3)")

train tokens: 22,621  test tokens: 8,491  (x 4 loops each)

point             linear probe   train   norm-only   per-loop recall (test)
after  0 layers         98.5%   99.7%       57.1%   100%  100%  96%  99%
after  6 layers         97.3%   99.0%       65.4%   100%  99%  94%  96%
after 18 layers         86.8%   92.7%       47.3%   97%  88%  70%  92%
after 24 layers         95.5%   98.3%       31.3%   99%  93%  95%  95%

chance = 25%. Confusion matrices (rows = true loop, cols = predicted), test set:
  after 0 layers: [[8491, 0, 0, 0], [23, 8449, 19, 0], [21, 328, 8113, 29], [13, 0, 92, 8386]]
  after 6 layers: [[8490, 1, 0, 0], [43, 8409, 27, 12], [65, 178, 7968, 280], [2, 14, 292, 8183]]
  after 18 layers: [[8276, 192, 1, 22], [780, 7503, 144, 64], [198, 843, 5915, 1535], [22, 40, 646, 7783]]
  after 24 layers: [[8417, 74, 0, 0], [592, 7866, 27, 6], [4, 75, 8046, 366], [1, 1, 388, 8101]]

saved probes -> /content/ouro_acts/loop_probes.pt (usage: logits = W @ ((h - mu) / sd) + b, argma

In [19]:
# ---------- Push the recorded activations, probes and probe results to the same HF dataset repo ----------
pr = probe_results
probe_md = f"""
## Loop-detection linear probes (`probe/`)

Residual stream recorded after 0, 6, 18 and 24 layers in every loop for {N:,} tokens
({len(seqs)} prompt+response sequences from the banks above, teacher-forced, capped at {MAX_LEN} tokens).
"After 0 layers" is the raw embedding in loop 1 and RMSNorm(previous loop output) in loops 2-4, because Ouro
applies its final norm between loops. One 4-way logistic-regression probe per point, split by sequence
(every 4th sequence held out). "Norm-only" is a control probe that sees only log ||h||.

| point | linear probe (test) | train | norm-only control | per-loop recall (loops 1-4) |
|---|---|---|---|---|
""" + "\n".join(
    f"| after {k} layers | {pr[k]['test_acc']:.1%} | {pr[k]['train_acc']:.1%} | {pr[k]['norm_only_test_acc']:.1%} | "
    + " / ".join(f"{r:.0%}" for r in pr[k]["per_loop_recall"]) + " |" for k in POINTS) + f"""

Chance is {1/S:.0%}. Loop 1 is always easy to spot; confusions are almost all between adjacent later loops.

Files: `probe/after{{0,6,18,24}}.npy` are fp16 arrays of shape [tokens, loop, hidden] = [{N}, {S}, {H}];
`probe/meta.npz` gives seq_id, position, token_id and bank (0 = math, 1 = code) per token;
`probe/loop_probes.pt` holds per-point `weight`, `bias`, `mu`, `sd` (apply as `W @ ((h - mu) / sd) + b`, argmax = loop index 0-3);
`probe/probe_results.json` has the accuracies and confusion matrices.
"""
readme_path = f"{OUT}/README.md"
txt = open(readme_path).read()
if "## Loop-detection linear probes" not in txt:
    open(readme_path, "w").write(txt + probe_md)
if os.path.exists("/content/ouro_eval.ipynb"):
    shutil.copy("/content/ouro_eval.ipynb", f"{OUT}/ouro_eval.ipynb")

api.upload_file(path_or_fileobj=readme_path, path_in_repo="README.md", repo_id=REPO, repo_type="dataset",
                commit_message="Add loop-detection probe results")
api.upload_file(path_or_fileobj=f"{OUT}/ouro_eval.ipynb", path_in_repo="ouro_eval.ipynb", repo_id=REPO, repo_type="dataset",
                commit_message="Update notebook with activation recording + probes")
t0 = time.time()
api.upload_folder(folder_path=ACT_DIR, path_in_repo="probe", repo_id=REPO, repo_type="dataset",
                  commit_message="Residual-stream activations at 4 points x 4 loops, loop probes, results")
print(f"pushed {sorted(os.listdir(ACT_DIR))} to https://huggingface.co/datasets/{REPO}/tree/main/probe in {time.time()-t0:.0f}s")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ent/ouro_acts/after18.npy:   0%|          |  531kB /  510MB            

  ...ent/ouro_acts/after24.npy:   0%|          |  524kB /  510MB            

  ...tent/ouro_acts/after0.npy:   1%|          | 3.70MB /  510MB            

  ...tent/ouro_acts/after6.npy:   0%|          |  529kB /  510MB            

  /content/ouro_acts/meta.npz :   7%|6         | 28.0kB /  405kB            

  .../ouro_acts/loop_probes.pt:   7%|6         | 13.9kB /  202kB            

pushed ['after0.npy', 'after18.npy', 'after24.npy', 'after6.npy', 'loop_probes.pt', 'meta.npz', 'probe_results.json'] to https://huggingface.co/datasets/mild-rgb/ouro-1.4b-thinking-evals/tree/main/probe in 33s


In [20]:
# ---------- Logit lens per loop, for every token of one chain of thought ----------
# Ouro applies its final RMSNorm at the end of each loop and feeds that to the early-exit gate, so
# lm_head(norm(h_loop_k)) is exactly what the model would have predicted had it stopped after loop k.
import torch, numpy as np

r = min([r for r in math_results if r["correct"]], key=lambda r: len(r["response"]))   # shortest correct CoT
print("QUESTION:", r["question"], f"\n(gold = {r['gold']:g})\n")
print("CHAIN OF THOUGHT (model output, re-tokenised):\n" + r["response"] + "\n" + "=" * 100)

prompt_ids = tokenizer.apply_chat_template(
    [{"role": "system", "content": MATH_SYS}, {"role": "user", "content": r["question"]}],
    tokenize=True, add_generation_prompt=True, return_tensors="pt")[0]
resp_ids = tokenizer(r["response"], add_special_tokens=False, return_tensors="pt")["input_ids"][0]
ids = torch.cat([prompt_ids, resp_ids]); P, T_all = len(prompt_ids), len(ids)

normed, gates = [], []
h1 = model.model.norm.register_forward_hook(lambda m, i, o: normed.append(o[0].detach()))
h2 = model.model.early_exit_gate.register_forward_hook(lambda m, i, o: gates.append(o[0].detach().float().squeeze(-1)))
try:
    with torch.no_grad():
        model(input_ids=ids[None].to(model.device), use_cache=False)
finally:
    h1.remove(); h2.remove()
assert len(normed) == S, f"expected {S} per-loop states, got {len(normed)}"

# per loop: log-probs over the vocab at every position (fp32), then the next-token stats
logp = [torch.log_softmax(model.lm_head(h).float(), -1) for h in normed]        # S x [T, V]
gate_p = [torch.sigmoid(g) for g in gates]                                       # S x [T]
tgt = ids[1:].to(model.device)                                                   # token predicted at position t is ids[t+1]
rows = []
for t in range(P - 1, T_all - 1):        # positions that predict the generated tokens
    row = {"pos": t + 1 - P, "actual": ids[t + 1].item(), "loops": []}
    for s in range(S):
        lp = logp[s][t]
        top = lp.argmax().item()
        rank = int((lp > lp[tgt[t]]).sum().item()) + 1
        row["loops"].append({"top": top, "p_actual": lp[tgt[t]].exp().item(), "rank": rank,
                             "gate": gate_p[s][t].item() if s < len(gate_p) else float("nan")})
    rows.append(row)

def show(tok_id, w=11):
    s = tokenizer.decode([tok_id]).replace("\n", "⏎").replace(" ", "␣")
    return (s[:w-1] + "…") if len(s) > w else s

print(f"\nPer-token logit lens. Cell = loop's top-1 token (p assigned to the actual token); '*' = top-1 matches the actual token.")
print(f"{'#':>4} {'actual':<12}" + "".join(f"{'loop '+str(s+1):<22}" for s in range(S)))
for row in rows:
    line = f"{row['pos']:>4} {show(row['actual']):<12}"
    for l in row["loops"]:
        mark = "*" if l["top"] == row["actual"] else " "
        line += f"{mark}{show(l['top'], 10):<10}({l['p_actual']:.2f})   "
    print(line)

# summary
print("\n" + "=" * 100 + "\nSUMMARY over the %d generated tokens" % len(rows))
final_top = [row["loops"][-1]["top"] for row in rows]
print(f"{'loop':<8}{'top-1 = actual':>16}{'mean p(actual)':>16}{'median rank':>13}{'agrees w/ loop 4':>18}{'mean exit gate':>16}")
for s in range(S):
    acc = np.mean([row["loops"][s]["top"] == row["actual"] for row in rows])
    mp = np.mean([row["loops"][s]["p_actual"] for row in rows])
    mr = np.median([row["loops"][s]["rank"] for row in rows])
    ag = np.mean([row["loops"][s]["top"] == ft for row, ft in zip(rows, final_top)])
    g = np.nanmean([row["loops"][s]["gate"] for row in rows])
    print(f"{s+1:<8}{acc:>16.1%}{mp:>16.3f}{mr:>13.0f}{ag:>18.1%}{g:>16.3f}")
conv = [next((s for s in range(S) if all(row["loops"][k]["top"] == final_top[i] for k in range(s, S))), S - 1) + 1
        for i, row in enumerate(rows)]
print("\nloop at which the top-1 prediction settles on its final value:",
      {f"loop {k}": int(np.sum(np.array(conv) == k)) for k in range(1, S + 1)})
changed = [(row["pos"], row) for row in rows if row["loops"][0]["top"] != row["loops"][-1]["top"]]
print(f"\n{len(changed)} tokens where loop 1 and loop 4 disagree on the top-1 prediction, e.g.:")
for pos, row in changed[:12]:
    print(f"  #{pos:<4} actual={show(row['actual'])!s:<12} " + " -> ".join(show(l["top"]) for l in row["loops"]))

QUESTION: Johnny took his allowance of $20 and added an extra $10 to it.  He then invested this sum of money, which tripled in a year.  How much money did he have after a year? 
(gold = 90)

CHAIN OF THOUGHT (model output, re-tokenised):

Okay, let's see. Johnny starts with $20 in allowance. Then he adds an extra $10. So first, I need to calculate the total amount he has after adding the extra money. That would be $20 plus $10, which is $30. Right?

Then he invests this $30, and the problem says the investment tripled in a year. Tripling means multiplying by 3. So $30 times 3. Let me do that math. 30 times 3 is 90. So after a year, the investment would be $90. 

Wait, let me double-check. Original amount after adding $10 is $30. Tripling that gives 30 * 3 = 90. Yep, that seems right. So the final amount he has after a year is $90.


Johnny starts with $20. Adding $10 gives him $30. Tripling this amount results in $30 × 3 = $90.

Final answer: 90

Per-token logit lens. Cell = loop's top

In [23]:
# ---------- Save the per-loop logit lens for this chain of thought and push it ----------
lens = {"question": r["question"], "gold": r["gold"], "response": r["response"],
        "prompt_tokens": P, "generated_tokens": len(rows),
        "columns": "per generated token: actual token id; per loop: top-1 token id, p(actual), rank of actual, exit-gate prob",
        "tokens": [{"pos": row["pos"], "actual": row["actual"], "actual_str": tokenizer.decode([row["actual"]]),
                    "loops": [{**l, "top_str": tokenizer.decode([l["top"]])} for l in row["loops"]]} for row in rows]}
json.dump(lens, open(f"{OUT}/logit_lens_example.json", "w"), indent=1)
if os.path.exists("/content/ouro_eval.ipynb"):
    shutil.copy("/content/ouro_eval.ipynb", f"{OUT}/ouro_eval.ipynb")

lens_md = """
## Per-loop logit lens on one chain of thought (`logit_lens_example.json`)

`lm_head(norm(h))` read at the end of each loop for every generated token of one correct GSM8K answer, i.e. what the
model would have said had it exited after that loop. Top-1 accuracy vs the actual next token by loop:
""" + " / ".join(f"loop {s+1}: {np.mean([row['loops'][s]['top'] == row['actual'] for row in rows]):.0%}" for s in range(S)) + \
f". Loop 1 already matches the final loop on {np.mean([row['loops'][0]['top'] == row['loops'][-1]['top'] for row in rows]):.0%} of tokens; " \
"later loops mostly repair function words and clause openers, not digits."
txt = open(f"{OUT}/README.md").read()
if "Per-loop logit lens" not in txt:
    open(f"{OUT}/README.md", "w").write(txt + lens_md)

for fn in ("README.md", "logit_lens_example.json", "ouro_eval.ipynb"):
    api.upload_file(path_or_fileobj=f"{OUT}/{fn}", path_in_repo=fn, repo_id=REPO, repo_type="dataset",
                    commit_message=f"Add per-loop logit lens example ({fn})")
print(f"pushed logit lens example + notebook to https://huggingface.co/datasets/{REPO}")

pushed logit lens example + notebook to https://huggingface.co/datasets/mild-rgb/ouro-1.4b-thinking-evals


In [24]:
# ---------- Per-loop logit lens over ALL bank answers (correct and incorrect) ----------
import json, os, re, time, numpy as np, torch

LENS_DIR = "/content/lens"; os.makedirs(LENS_DIR, exist_ok=True)

def kind_of(s):
    t = s.strip()
    if t == "": return "space"
    if t.isdigit(): return "digit"
    if re.fullmatch(r"[^\w\s]+", t): return "punct"
    return "word"

def lens_for(system, user, response, meta):
    prompt_ids = tokenizer.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}],
                                               tokenize=True, add_generation_prompt=True, return_tensors="pt")[0]
    resp_ids = tokenizer(response, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    ids = torch.cat([prompt_ids, resp_ids])[:MAX_LEN]; P = len(prompt_ids)
    if len(ids) <= P + 1: return None, None
    normed, gates = [], []
    h1 = model.model.norm.register_forward_hook(lambda m, i, o: normed.append(o[0].detach()))
    h2 = model.model.early_exit_gate.register_forward_hook(lambda m, i, o: gates.append(o[0].detach().float().squeeze(-1)))
    try:
        with torch.no_grad():
            model(input_ids=ids[None].to(model.device), use_cache=False)
            pos = torch.arange(P - 1, len(ids) - 1, device=model.device)      # positions predicting generated tokens
            tgt = ids[1:].to(model.device)[pos]
            per_loop = []
            for s in range(S):
                lp = torch.log_softmax(model.lm_head(normed[s][pos]).float(), -1)     # [n, V]
                per_loop.append(lp)
            final = per_loop[-1]
            recs = []
            n = len(pos)
            cols = {}
            for s, lp in enumerate(per_loop):
                p_act = lp.gather(1, tgt[:, None])[:, 0]
                cols[s] = dict(top=lp.argmax(1).cpu(), p=p_act.exp().cpu(),
                               rank=((lp > p_act[:, None]).sum(1) + 1).cpu(),
                               ent=(-(lp.exp() * lp).sum(1)).cpu(),
                               kl=((final.exp() * (final - lp)).sum(1)).cpu(),           # KL(loop4 || loop s)
                               gate=torch.sigmoid(gates[s][pos]).cpu())
            del per_loop, final
    finally:
        h1.remove(); h2.remove()
    ids_c = ids.cpu()
    for i in range(n):
        a = ids_c[pos[i] + 1].item(); a_str = tokenizer.decode([a])
        rec = {**meta, "pos": i, "actual": a, "actual_str": a_str, "kind": kind_of(a_str),
               "top": [cols[s]["top"][i].item() for s in range(S)],
               "p": [round(cols[s]["p"][i].item(), 4) for s in range(S)],
               "rank": [cols[s]["rank"][i].item() for s in range(S)],
               "ent": [round(cols[s]["ent"][i].item(), 3) for s in range(S)],
               "kl_to_final": [round(cols[s]["kl"][i].item(), 4) for s in range(S)],
               "gate": [round(cols[s]["gate"][i].item(), 3) for s in range(S)]}
        recs.append(rec)
    return recs, {"prompt_tokens": P, "generated_tokens": n}

jobs = []
for i, r in enumerate(math_results):
    jobs.append((MATH_SYS, r["question"], r["response"],
                 {"seq": f"math{i}", "bank": "math", "correct": bool(r["correct"]), "truncated": bool(r["truncated"]),
                  "pred": r["pred"], "gold": r["gold"]}))
for i, (r, ex) in enumerate(zip(code_results, mbpp)):
    jobs.append((CODE_SYS, code_prompt(ex), r["response"],
                 {"seq": f"code{i}", "bank": "code", "correct": bool(r["passed"]), "truncated": not r["had_code_block"],
                  "task_id": ex["task_id"], "error": r["error"]}))

t0 = time.time(); seq_summ = []
with open(f"{LENS_DIR}/lens_tokens.jsonl", "w") as f:
    for j, (sys_, user, resp, meta) in enumerate(jobs):
        recs, info = lens_for(sys_, user, resp, meta)
        if recs is None: continue
        for rec in recs: f.write(json.dumps(rec) + "\n")
        summ = {**meta, **info, "response": resp}
        for s in range(S):
            summ[f"loop{s+1}"] = {
                "top1_acc": float(np.mean([rc["top"][s] == rc["actual"] for rc in recs])),
                "mean_p": float(np.mean([rc["p"][s] for rc in recs])),
                "mean_logp": float(np.mean([np.log(max(rc["p"][s], 1e-9)) for rc in recs])),
                "agree_final": float(np.mean([rc["top"][s] == rc["top"][-1] for rc in recs])),
                "mean_gate": float(np.mean([rc["gate"][s] for rc in recs])),
                "mean_ent": float(np.mean([rc["ent"][s] for rc in recs])),
                "mean_kl_to_final": float(np.mean([rc["kl_to_final"][s] for rc in recs])),
                "digit_top1_acc": float(np.mean([rc["top"][s] == rc["actual"] for rc in recs if rc["kind"] == "digit"] or [np.nan])),
                "word_top1_acc": float(np.mean([rc["top"][s] == rc["actual"] for rc in recs if rc["kind"] == "word"] or [np.nan])),
            }
        settle = [next((s for s in range(S) if all(rc["top"][k] == rc["top"][-1] for k in range(s, S))), S - 1) + 1 for rc in recs]
        summ["settle_hist"] = {f"loop{k}": int(np.sum(np.array(settle) == k)) for k in range(1, S + 1)}
        seq_summ.append(summ)
        print(f"  {meta['seq']:<7} {'ok ' if meta['correct'] else 'BAD'} {'trunc ' if meta['truncated'] else '      '}"
              f"{info['generated_tokens']:>5} tok  top1 by loop: " + " ".join(f"{summ[f'loop{s+1}']['top1_acc']:.0%}" for s in range(S))
              + f"   gate4={summ['loop4']['mean_gate']:.2f}  {time.time()-t0:.0f}s")
json.dump(seq_summ, open(f"{LENS_DIR}/lens_sequences.json", "w"), indent=1)

# pooled table by (bank, correct)
print("\nPooled per-loop top-1 accuracy / mean p(actual) / mean exit gate, by group:")
for bank in ("math", "code"):
    for corr in (True, False):
        grp = [x for x in seq_summ if x["bank"] == bank and x["correct"] == corr]
        if not grp: continue
        w = np.array([x["generated_tokens"] for x in grp], float)
        line = f"  {bank:<5} {'correct  ' if corr else 'incorrect'} n={len(grp):<3} tokens={int(w.sum()):>6} | "
        for s in range(S):
            acc = np.average([x[f'loop{s+1}']['top1_acc'] for x in grp], weights=w)
            mp = np.average([x[f'loop{s+1}']['mean_p'] for x in grp], weights=w)
            g = np.average([x[f'loop{s+1}']['mean_gate'] for x in grp], weights=w)
            line += f"L{s+1}: {acc:.1%}/{mp:.2f}/{g:.2f}  "
        print(line)
sz = sum(os.path.getsize(f"{LENS_DIR}/{f}") for f in os.listdir(LENS_DIR)) / 2**20
print(f"\nwrote {LENS_DIR}/lens_tokens.jsonl + lens_sequences.json ({sz:.0f} MB) in {time.time()-t0:.0f}s")

  math0   ok          508 tok  top1 by loop: 89% 92% 94% 94%   gate4=0.49  1s
  math1   ok  trunc   297 tok  top1 by loop: 85% 91% 91% 91%   gate4=0.49  1s
  math2   BAD trunc   913 tok  top1 by loop: 84% 92% 92% 93%   gate4=0.48  2s
  math3   ok          422 tok  top1 by loop: 87% 92% 94% 94%   gate4=0.49  3s
  math4   ok          509 tok  top1 by loop: 88% 92% 93% 94%   gate4=0.49  4s
  math5   BAD trunc   917 tok  top1 by loop: 79% 89% 91% 91%   gate4=0.48  5s
  math6   BAD trunc   131 tok  top1 by loop: 91% 92% 92% 94%   gate4=0.50  5s
  math7   ok          351 tok  top1 by loop: 81% 88% 92% 92%   gate4=0.49  6s
  math8   BAD trunc   947 tok  top1 by loop: 88% 92% 93% 94%   gate4=0.49  7s
  math9   ok          794 tok  top1 by loop: 82% 90% 92% 92%   gate4=0.49  8s
  math10  ok          233 tok  top1 by loop: 78% 87% 89% 88%   gate4=0.49  8s
  math11  BAD trunc   934 tok  top1 by loop: 84% 89% 90% 91%   gate4=0.48  10s
  math12  ok          862 tok  top1 by loop: 84% 89% 91% 91%   

In [65]:
# ---------- Push the full logit-lens sweep to the HF repo ----------
api.upload_folder(folder_path=LENS_DIR, path_in_repo="lens", repo_id=REPO, repo_type="dataset",
                  commit_message="Per-loop logit lens over all 40 bank answers (sampled, T=1.0 top_p=0.7)")
print(f"pushed {sorted(os.listdir(LENS_DIR))} to https://huggingface.co/datasets/{REPO}/tree/main/lens")

pushed ['lens_sequences.json', 'lens_tokens.jsonl'] to https://huggingface.co/datasets/mild-rgb/ouro-1.4b-thinking-evals/tree/main/lens


In [71]:
# ---------- Push the subagent analysis reports + scripts alongside the lens data ----------
api.upload_folder(folder_path="/content/lens/analysis", path_in_repo="lens/analysis", repo_id=REPO, repo_type="dataset",
                  commit_message="Sonnet 5 analyses of the per-loop logit lens sweep (3 reports + scripts)")
print(f"pushed {sorted(os.listdir('/content/lens/analysis'))} to https://huggingface.co/datasets/{REPO}/tree/main/lens/analysis")

pushed ['analyze.py', 'cvi_analyze.py', 'report_code_vs_math_truncation.md', 'report_correct_vs_incorrect.md', 'report_token_dynamics.md'] to https://huggingface.co/datasets/mild-rgb/ouro-1.4b-thinking-evals/tree/main/lens/analysis


In [73]:
# ---------- Contrastive Activation Addition (CAA) on Ouro, sentiment: which loop to steer? ----------
# Steering vector = mean residual at the final token of "The <noun> was <positive adj>" minus the same for a negative adj.
# Test = held-out neutral prompts; metric = p(positive adjectives) / (p(positive) + p(negative)) for the next token.
import json, time, numpy as np, torch

NOUNS = ["movie", "meal", "hotel", "concert", "book", "service", "flight", "game", "lecture", "party",
         "trip", "museum", "show", "phone", "app", "coffee", "teacher", "doctor", "weather", "garden"]
POS = ["wonderful", "great", "amazing", "excellent", "fantastic", "delightful", "lovely", "superb", "brilliant", "perfect", "good", "awesome", "incredible", "pleasant", "enjoyable"]
NEG = ["terrible", "awful", "horrible", "disappointing", "dreadful", "bad", "poor", "boring", "miserable", "frustrating", "mediocre", "lousy", "painful", "unpleasant", "appalling"]
single = lambda w: len(tokenizer.encode(" " + w, add_special_tokens=False)) == 1
POS, NEG = [w for w in POS if single(w)], [w for w in NEG if single(w)]
POS_IDS = [tokenizer.encode(" " + w, add_special_tokens=False)[0] for w in POS]
NEG_IDS = [tokenizer.encode(" " + w, add_special_tokens=False)[0] for w in NEG]
print(f"{len(POS)} positive / {len(NEG)} negative single-token adjectives")

CAP_LAYERS = {6: 5, 12: 11, 18: 17}          # "after k layers" -> decoder layer index whose output we use
layers = model.model.layers
tokenizer.padding_side = "left"

# ---- 1. Extract steering vectors at the final (adjective) token, per capture point and loop
EXTRACT_TEMPLATES = ["The {n} was", "I thought the {n} was", "In the end the {n} was"]
texts, labels = [], []
for n in NOUNS:
    for tmpl in EXTRACT_TEMPLATES:
        for i in range(8):
            texts += [tmpl.format(n=n) + " " + POS[i], tmpl.format(n=n) + " " + NEG[i]]; labels += [1, 0]
sums = {(k, s): torch.zeros(2, H, dtype=torch.float32, device=model.device) for k in CAP_LAYERS for s in range(S)}
norms = {(k, s): [0.0, 0] for k in CAP_LAYERS for s in range(S)}
cur_batch_label = None
def make_cap_hook(k):
    def hook(mod, args, kwargs, out):
        h = out[0] if isinstance(out, tuple) else out
        s = int(kwargs["current_ut"]); last = h[:, -1].float()               # left-padded, so -1 is the adjective
        for i, lab in enumerate(cur_batch_label): sums[(k, s)][lab] += last[i]
        norms[(k, s)][0] += last.norm(dim=-1).sum().item(); norms[(k, s)][1] += len(last)
    return hook
handles = [layers[idx].register_forward_hook(make_cap_hook(k), with_kwargs=True) for k, idx in CAP_LAYERS.items()]
t0 = time.time()
try:
    with torch.no_grad():
        for i in range(0, len(texts), 32):
            enc = tokenizer(texts[i:i+32], return_tensors="pt", padding=True).to(model.device)
            cur_batch_label = labels[i:i+32]; model(**enc, use_cache=False)
finally:
    for h in handles: h.remove()
    handles = []
n_pos = sum(labels); n_neg = len(labels) - n_pos
vecs = {key: (sums[key][1] / n_pos - sums[key][0] / n_neg) for key in sums}
print(f"extracted {len(vecs)} steering vectors from {n_pos} contrast pairs in {time.time()-t0:.0f}s")
print(f"{'point':<10}" + "".join(f"{'loop '+str(s+1):>26}" for s in range(S)))
for k in CAP_LAYERS:
    print(f"after {k:>2}   " + "".join(f"  |v|={vecs[(k,s)].norm():5.2f} / |h|={norms[(k,s)][0]/norms[(k,s)][1]:5.1f}" for s in range(S)))
cos = lambda a, b: torch.nn.functional.cosine_similarity(a, b, dim=0).item()
print("cosine between vectors extracted at different loops (same point):")
for k in CAP_LAYERS:
    print(f"  after {k:>2}: " + "  ".join(f"L{a+1}·L{b+1}={cos(vecs[(k,a)], vecs[(k,b)]):.2f}" for a in range(S) for b in range(a+1, S)))

# ---- 2. Evaluate on held-out neutral prompts
TEST_TEMPLATES = ["Honestly, I thought the {n} was", "Everyone agreed that the {n} was",
                  "After all that, the {n} turned out to be", "My overall impression of the {n} is that it was"]
test_texts = [t.format(n=n) for n in NOUNS for t in TEST_TEMPLATES]
steer_cfg = {}    # {layer_idx: {loop: vector}}
def make_steer_hook(idx):
    def hook(mod, args, kwargs, out):
        v = steer_cfg.get(idx, {}).get(int(kwargs["current_ut"]))
        if v is None: return None
        h = out[0] if isinstance(out, tuple) else out
        h2 = h + v.to(h.dtype)
        return (h2,) + tuple(out[1:]) if isinstance(out, tuple) else h2
    return hook

@torch.no_grad()
def eval_sentiment(bs=16):
    ps = []
    for i in range(0, len(test_texts), bs):
        enc = tokenizer(test_texts[i:i+bs], return_tensors="pt", padding=True).to(model.device)
        p = torch.softmax(model(**enc, use_cache=False).logits[:, -1].float(), -1)
        pp, pn = p[:, POS_IDS].sum(1), p[:, NEG_IDS].sum(1)
        ps += (pp / (pp + pn)).tolist()
    return float(np.mean(ps))

handles = [layers[idx].register_forward_hook(make_steer_hook(idx), with_kwargs=True) for idx in CAP_LAYERS.values()]
results = []
try:
    steer_cfg.clear(); base_p = eval_sentiment()
    print(f"\nBASELINE: p(positive | pos or neg adjective) = {base_p:.3f} over {len(test_texts)} prompts")
    MULTS = [-2, -1, 1, 2]
    t0 = time.time()
    for k, idx in CAP_LAYERS.items():
        for loops_name, loops in [("L1", [0]), ("L2", [1]), ("L3", [2]), ("L4", [3]), ("all", [0, 1, 2, 3])]:
            for m in MULTS:
                steer_cfg.clear(); steer_cfg[idx] = {s: m * vecs[(k, s)] for s in loops}   # each loop gets its own vector
                results.append({"point": k, "inject": loops_name, "vector": "same-loop", "mult": m, "p": eval_sentiment()})
        print(f"  after {k} layers done ({time.time()-t0:.0f}s)")
    for src, dst in [(0, 3), (3, 0), (0, 1), (2, 3)]:            # transfer: vector from one loop injected at another
        for m in [-2, 2]:
            steer_cfg.clear(); steer_cfg[CAP_LAYERS[12]] = {dst: m * vecs[(12, src)]}
            results.append({"point": 12, "inject": f"L{dst+1}", "vector": f"from L{src+1}", "mult": m, "p": eval_sentiment()})
    g = torch.Generator().manual_seed(0)                          # control: random direction, same norm, all loops
    for m in [-2, 2]:
        steer_cfg.clear()
        steer_cfg[CAP_LAYERS[12]] = {s: m * (lambda r: r / r.norm() * vecs[(12, s)].norm())(torch.randn(H, generator=g).to(model.device)) for s in range(S)}
        results.append({"point": 12, "inject": "all", "vector": "random", "mult": m, "p": eval_sentiment()})

    # ---- 3. Report
    print(f"\nCAA sentiment: mean p(positive) on {len(test_texts)} held-out prompts (baseline {base_p:.3f}). Rows: injection loop; cols: multiplier.")
    for k in CAP_LAYERS:
        print(f"\nvector extracted & injected after {k} layers")
        print(f"{'inject':<8}" + "".join(f"{'x'+str(m):>10}" for m in MULTS) + f"{'  span(-2..+2)':>16}")
        for ln in ["L1", "L2", "L3", "L4", "all"]:
            rr = {r["mult"]: r["p"] for r in results if r["point"] == k and r["inject"] == ln and r["vector"] == "same-loop"}
            print(f"{ln:<8}" + "".join(f"{rr[m]:>10.3f}" for m in MULTS) + f"{rr[2]-rr[-2]:>16.3f}")
    print("\ntransfer (after 12 layers) and control:")
    for r in results:
        if r["vector"] != "same-loop":
            print(f"  vector {r['vector']:<9} injected at {r['inject']:<4} x{r['mult']:+d}: p={r['p']:.3f}")

    # ---- 4. Qualitative: greedy generations at the best single-loop config
    best = max((r for r in results if r["vector"] == "same-loop" and r["mult"] == 2), key=lambda r: r["p"])
    k, ln = best["point"], best["inject"]; loops = [0, 1, 2, 3] if ln == "all" else [int(ln[1]) - 1]
    print(f"\nGreedy continuations, steering after {k} layers at {ln}:")
    gen_prompt = "Let me tell you about the restaurant we went to last night. The"
    enc = tokenizer(gen_prompt, return_tensors="pt").to(model.device)
    for m in [-2, 0, 2]:
        steer_cfg.clear()
        if m: steer_cfg[CAP_LAYERS[k]] = {s: m * vecs[(k, s)] for s in loops}
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        print(f"  x{m:+d}: {tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)!r}")
finally:
    for h in handles: h.remove()
    handles = []; steer_cfg.clear()

json.dump({"behavior": "sentiment", "baseline_p": base_p, "n_test": len(test_texts), "n_pairs": n_pos, "pos": POS, "neg": NEG,
           "vector_norms": {f"after{k}_L{s+1}": vecs[(k, s)].norm().item() for k in CAP_LAYERS for s in range(S)},
           "resid_norms": {f"after{k}_L{s+1}": norms[(k, s)][0] / norms[(k, s)][1] for k in CAP_LAYERS for s in range(S)},
           "results": results}, open(f"{OUT}/caa_sentiment.json", "w"), indent=1)
torch.save({f"after{k}_L{s+1}": vecs[(k, s)].cpu() for k in CAP_LAYERS for s in range(S)}, f"{OUT}/caa_sentiment_vectors.pt")
print(f"\nsaved {OUT}/caa_sentiment.json and caa_sentiment_vectors.pt")

15 positive / 12 negative single-token adjectives
extracted 12 steering vectors from 480 contrast pairs in 7s
point                         loop 1                    loop 2                    loop 3                    loop 4
after  6     |v|= 1.80 / |h|=  3.8  |v|= 2.85 / |h|=  6.3  |v|= 3.74 / |h|=  7.8  |v|= 4.00 / |h|=  8.3
after 12     |v|= 2.93 / |h|=  6.1  |v|= 4.32 / |h|=  8.8  |v|= 4.92 / |h|= 10.2  |v|= 5.00 / |h|= 10.5
after 18     |v|= 3.72 / |h|=  8.3  |v|= 5.20 / |h|= 11.3  |v|= 5.82 / |h|= 12.9  |v|= 5.90 / |h|= 13.2
cosine between vectors extracted at different loops (same point):
  after  6: L1·L2=0.84  L1·L3=0.83  L1·L4=0.83  L2·L3=0.97  L2·L4=0.96  L3·L4=0.99
  after 12: L1·L2=0.91  L1·L3=0.90  L1·L4=0.89  L2·L3=0.98  L2·L4=0.96  L3·L4=0.99
  after 18: L1·L2=0.92  L1·L3=0.91  L1·L4=0.90  L2·L3=0.99  L2·L4=0.98  L3·L4=1.00

BASELINE: p(positive | pos or neg adjective) = 0.781 over 80 prompts
  after 6 layers done (18s)
  after 12 layers done (35s)
  after 18 layers don

In [74]:
# ---------- CAA v2: calibrated strengths, random control everywhere, and a damage metric ----------
# The first sweep added vectors whose norm (x2) rivalled the residual stream itself, and a random direction of the same
# norm moved the metric nearly as far. Here the injection is alpha * |h|_(point,loop) * unit(v), so alpha is the fraction
# of the local residual norm, and every configuration is paired with a random direction of identical norm.
import json, time, numpy as np, torch

ALPHAS = [-0.4, -0.2, -0.1, 0.1, 0.2, 0.4]
resid = {key: norms[key][0] / norms[key][1] for key in norms}
unit = {key: vecs[key] / vecs[key].norm() for key in vecs}
g = torch.Generator().manual_seed(0)
rand_unit = {key: (lambda r: r / r.norm())(torch.randn(H, generator=g)).to(model.device) for key in vecs}

NEUTRAL = ("The Danube is the second-longest river in Europe. It rises in the Black Forest in Germany and flows "
           "through ten countries before reaching the Black Sea. Its basin covers parts of nineteen countries, and "
           "several capital cities, including Vienna, Bratislava, Budapest and Belgrade, lie on its banks. The river has "
           "been an important trade route since antiquity.")
neutral_enc = tokenizer(NEUTRAL, return_tensors="pt").to(model.device)

@torch.no_grad()
def neutral_nll():
    logits = model(**neutral_enc, use_cache=False).logits[0, :-1].float()
    return torch.nn.functional.cross_entropy(logits, neutral_enc["input_ids"][0, 1:]).item()

handles = [layers[idx].register_forward_hook(make_steer_hook(idx), with_kwargs=True) for idx in CAP_LAYERS.values()]
res2 = []
try:
    steer_cfg.clear(); base_p, base_nll = eval_sentiment(), neutral_nll()
    print(f"BASELINE: p(positive)={base_p:.3f}, neutral-text NLL={base_nll:.3f} nats/token")
    t0 = time.time()
    for k, idx in CAP_LAYERS.items():
        for ln, loops in [("L1", [0]), ("L2", [1]), ("L3", [2]), ("L4", [3]), ("all", [0, 1, 2, 3])]:
            for a in ALPHAS:
                for kind, U in (("caa", unit), ("random", rand_unit)):
                    steer_cfg.clear(); steer_cfg[idx] = {s: a * resid[(k, s)] * U[(k, s)] for s in loops}
                    res2.append({"point": k, "inject": ln, "vector": kind, "alpha": a, "p": eval_sentiment(), "nll": neutral_nll()})
        print(f"  after {k} layers done ({time.time()-t0:.0f}s)")

    print(f"\nCAA sentiment v2. Cell = p(positive) for CAA vector [random control]  |  baseline p={base_p:.3f}")
    for k in CAP_LAYERS:
        print(f"\nafter {k} layers        " + "".join(f"{'a='+str(a):>18}" for a in ALPHAS))
        for ln in ["L1", "L2", "L3", "L4", "all"]:
            row = f"{ln:<8}"
            for a in ALPHAS:
                c = next(r for r in res2 if r["point"] == k and r["inject"] == ln and r["alpha"] == a and r["vector"] == "caa")
                z = next(r for r in res2 if r["point"] == k and r["inject"] == ln and r["alpha"] == a and r["vector"] == "random")
                row += f"{c['p']:>10.3f} [{z['p']:.2f}]  "
            print(row)
    print(f"\nDamage: neutral-text NLL (baseline {base_nll:.2f}); CAA [random]")
    for k in CAP_LAYERS:
        print(f"\nafter {k} layers        " + "".join(f"{'a='+str(a):>18}" for a in ALPHAS))
        for ln in ["L1", "L2", "L3", "L4", "all"]:
            row = f"{ln:<8}"
            for a in ALPHAS:
                c = next(r for r in res2 if r["point"] == k and r["inject"] == ln and r["alpha"] == a and r["vector"] == "caa")
                z = next(r for r in res2 if r["point"] == k and r["inject"] == ln and r["alpha"] == a and r["vector"] == "random")
                row += f"{c['nll']:>10.2f} [{z['nll']:.2f}]  "
            print(row)

    # specificity score: |shift| of CAA minus |shift| of random, at alpha=+-0.2, per config
    print("\nSpecific effect at |alpha|=0.2 (CAA shift minus random shift, positive minus negative direction):")
    print(f"{'point':<10}" + "".join(f"{ln:>10}" for ln in ["L1", "L2", "L3", "L4", "all"]))
    for k in CAP_LAYERS:
        row = f"after {k:<4}"
        for ln in ["L1", "L2", "L3", "L4", "all"]:
            get = lambda kind, a: next(r["p"] for r in res2 if r["point"] == k and r["inject"] == ln and r["alpha"] == a and r["vector"] == kind)
            row += f"{(get('caa', 0.2) - get('caa', -0.2)) - (get('random', 0.2) - get('random', -0.2)):>10.3f}"
        print(row)

    # generations at a moderate strength, best specific single-loop config after 12 layers
    print("\nGreedy continuations (after 12 layers, all loops), alpha = -0.2 / 0 / +0.2 and +-0.4:")
    enc = tokenizer("Let me tell you about the restaurant we went to last night. The", return_tensors="pt").to(model.device)
    for a in [-0.4, -0.2, 0, 0.2, 0.4]:
        steer_cfg.clear()
        if a: steer_cfg[CAP_LAYERS[12]] = {s: a * resid[(12, s)] * unit[(12, s)] for s in range(S)}
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        print(f"  a={a:+.1f}: {tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)!r}")
finally:
    for h in handles: h.remove()
    handles = []; steer_cfg.clear()

json.dump({"behavior": "sentiment", "scaling": "alpha * local mean residual norm * unit(v)", "baseline_p": base_p, "baseline_nll": base_nll,
           "resid_norms": {f"after{k}_L{s+1}": resid[(k, s)] for k in CAP_LAYERS for s in range(S)}, "results": res2},
          open(f"{OUT}/caa_sentiment_v2.json", "w"), indent=1)
print(f"\nsaved {OUT}/caa_sentiment_v2.json")

BASELINE: p(positive)=0.781, neutral-text NLL=1.363 nats/token
  after 6 layers done (60s)
  after 12 layers done (119s)
  after 18 layers done (178s)

CAA sentiment v2. Cell = p(positive) for CAA vector [random control]  |  baseline p=0.781

after 6 layers                    a=-0.4            a=-0.2            a=-0.1             a=0.1             a=0.2             a=0.4
L1           0.635 [0.81]       0.700 [0.79]       0.741 [0.78]       0.820 [0.78]       0.855 [0.78]       0.933 [0.79]  
L2           0.526 [0.80]       0.655 [0.80]       0.716 [0.79]       0.828 [0.77]       0.864 [0.76]       0.917 [0.76]  
L3           0.341 [0.74]       0.526 [0.76]       0.655 [0.77]       0.865 [0.79]       0.914 [0.79]       0.964 [0.80]  
L4           0.096 [0.69]       0.234 [0.74]       0.493 [0.76]       0.927 [0.80]       0.976 [0.82]       0.997 [0.85]  
all          0.103 [0.66]       0.138 [0.75]       0.242 [0.77]       0.976 [0.80]       0.999 [0.81]       1.000 [0.85]  

after 12 l

In [77]:
# ---------- Does the loop contract a perturbation? Inject at one loop, track the difference through the later loops ----------
import numpy as np, torch, json

IDX = CAP_LAYERS[12]; ALPHA = 0.2
# readout direction at the end-of-loop (normed) state: mean unembedding of positive minus negative adjectives
W = model.lm_head.weight.detach().float()
w_read = W[POS_IDS].mean(0) - W[NEG_IDS].mean(0); w_read = w_read / w_read.norm()

caps = {}   # ("mid", loop) -> layer-12 output; ("end", loop) -> normed end-of-loop state; each [batch, T, H] float32 on CPU
def cap_mid(mod, args, kwargs, out):
    h = out[0] if isinstance(out, tuple) else out
    caps[("mid", int(kwargs["current_ut"]))] = h.detach().float().cpu()
end_counter = [0]
def cap_end(mod, inp, out):
    caps[("end", end_counter[0])] = out.detach().float().cpu(); end_counter[0] += 1

@torch.no_grad()
def run_capture(cfg):
    """Forward over the test prompts with steering cfg; returns dict of captured states concatenated over prompts."""
    steer_cfg.clear(); steer_cfg.update(cfg)
    acc = {}
    for i in range(0, len(test_texts), 16):
        enc = tokenizer(test_texts[i:i+16], return_tensors="pt", padding=True).to(model.device)
        caps.clear(); end_counter[0] = 0
        model(**enc, use_cache=False)
        mask = enc["attention_mask"].bool().cpu()
        for key, h in caps.items():
            acc.setdefault(key, []).append((h[mask], h[:, -1]))     # (all real tokens, last token)
    return {k: (torch.cat([a for a, _ in v]), torch.cat([b for _, b in v])) for k, v in acc.items()}

# steering hooks first, capture hooks second, so the layer-12 capture sees the state AFTER the vector is added
h_steer = [layers[idx].register_forward_hook(make_steer_hook(idx), with_kwargs=True) for idx in CAP_LAYERS.values()]
h_mid = [layers[IDX].register_forward_hook(cap_mid, with_kwargs=True), model.model.norm.register_forward_hook(cap_end)]
try:
    base = run_capture({})
    rows = []
    for inj in (0, 1, 2):
        for kind, U in (("caa", unit), ("random", rand_unit)):
            for sign in (+1, -1):
                v_inj = sign * ALPHA * resid[(12, inj)] * U[(12, inj)]
                st = run_capture({IDX: {inj: v_inj}})
                inj_norm = v_inj.norm().item()
                for s in range(inj, S):
                    for where in ("mid", "end"):
                        for which, j in (("all", 0), ("last", 1)):
                            d = st[(where, s)][j] - base[(where, s)][j]                 # [n, H]
                            dn = d.norm(dim=1)
                            direction = unit[(12, s)].cpu() if where == "mid" else w_read.cpu()
                            proj = d @ direction                                        # signed component along the direction
                            rows.append({"inject_loop": inj + 1, "kind": kind, "sign": sign, "loop": s + 1, "where": where, "tokens": which,
                                         "rel_norm": (dn.mean() / inj_norm).item(),
                                         "rel_proj": (proj.mean() / inj_norm).item(),
                                         "along_frac": (proj.abs().mean() / dn.mean()).item()})
finally:
    for h in h_mid + h_steer: h.remove()
    handles = []; steer_cfg.clear()

def show(where, tokens):
    print(f"\n[{where} = {'layer-12 output' if where=='mid' else 'end-of-loop normed state'}, {tokens} tokens]   "
          f"cells: |delta| / |injected|   (signed component along the {'CAA' if where=='mid' else 'readout'} direction / |injected|)")
    print(f"{'inject@':<9}{'kind':<8}" + "".join(f"{'loop '+str(s+1):>22}" for s in range(S)))
    for inj in (1, 2, 3):
        for kind in ("caa", "random"):
            line = f"L{inj:<8}{kind:<8}"
            for s in range(1, S + 1):
                rr = [r for r in rows if r["inject_loop"] == inj and r["kind"] == kind and r["loop"] == s and r["where"] == where and r["tokens"] == tokens]
                if not rr: line += f"{'':>22}"; continue
                rn = np.mean([r["rel_norm"] for r in rr]); rp = np.mean([r["sign"] * r["rel_proj"] for r in rr])
                line += f"{rn:>10.2f} ({rp:+.2f})   "
            print(line)
show("mid", "all"); show("mid", "last"); show("end", "all"); show("end", "last")

print("\nPer-loop decay of |delta| and of its on-direction component at the layer-12 output (all tokens), injected at loop 1:")
for kind in ("caa", "random"):
    rr = lambda s, f: np.mean([r[f] * (r["sign"] if f == "rel_proj" else 1) for r in rows if r["inject_loop"] == 1 and r["kind"] == kind and r["loop"] == s and r["where"] == "mid" and r["tokens"] == "all"])
    nrm = [rr(s, "rel_norm") for s in range(1, S + 1)]; prj = [rr(s, "rel_proj") for s in range(1, S + 1)]
    print(f"  {kind:<7} |delta|: " + " -> ".join(f"{x:.2f}" for x in nrm) + "   ratios: " + ", ".join(f"{nrm[i+1]/nrm[i]:.2f}" for i in range(S - 1)))
    print(f"  {'':<7} along : " + " -> ".join(f"{x:+.2f}" for x in prj) + "   ratios: " + ", ".join(f"{prj[i+1]/prj[i]:.2f}" if abs(prj[i]) > 1e-6 else "n/a" for i in range(S - 1)))
print("\nReadout-direction component at the end-of-loop state (last token), by injection loop:")
for inj in (1, 2, 3):
    prj = [np.mean([r["sign"] * r["rel_proj"] for r in rows if r["inject_loop"] == inj and r["kind"] == "caa" and r["loop"] == s and r["where"] == "end" and r["tokens"] == "last"]) for s in range(inj, S + 1)]
    print(f"  inject at L{inj}: " + " -> ".join(f"{x:+.2f}" for x in prj) + "   ratios: " + ", ".join(f"{prj[i+1]/prj[i]:.2f}" for i in range(len(prj) - 1)))
json.dump(rows, open(f"{OUT}/caa_perturbation_decay.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{OUT}/caa_perturbation_decay.json", path_in_repo="caa/caa_perturbation_decay.json",
                repo_id=REPO, repo_type="dataset", commit_message="Perturbation decay through Ouro's loops (fixed capture order)")
print("\nsaved + pushed caa/caa_perturbation_decay.json")


[mid = layer-12 output, all tokens]   cells: |delta| / |injected|   (signed component along the CAA direction / |injected|)
inject@  kind                    loop 1                loop 2                loop 3                loop 4
L1       caa           1.00 (+1.00)         1.33 (+0.24)         1.25 (+0.17)         1.00 (+0.14)   
L1       random        1.00 (+0.06)         0.71 (-0.00)         0.60 (-0.01)         0.47 (-0.01)   
L2       caa                                 1.00 (+1.00)         0.92 (+0.23)         0.67 (+0.15)   
L2       random                              1.00 (+0.03)         0.55 (-0.01)         0.41 (-0.01)   
L3       caa                                                       1.00 (+1.00)         0.75 (+0.25)   
L3       random                                                    1.00 (-0.01)         0.39 (+0.00)   

[mid = layer-12 output, last tokens]   cells: |delta| / |injected|   (signed component along the CAA direction / |injected|)
inject@  kind            

In [78]:
# ---------- CAA across concepts: same recipe (extract after 12 layers per loop, sweep injection loop, random control, damage, decay) ----------
import json, time, numpy as np, torch

CTX = ["yesterday", "at the park", "in the garden", "near the river", "at school", "in town", "on the road", "at the market",
       "by the lake", "in the city", "on the farm", "at the beach", "in the forest", "at home", "in the shop", "on the hill"]
SUBJ = ["the dog", "my sister", "the teacher", "our neighbour", "the little boy", "a stranger", "the old man", "her friend",
        "the cat", "the driver", "the nurse", "the pilot", "a child", "the farmer", "my uncle", "the girl"]
YN_EXTRACT = ["Is water wet?", "Can birds fly?", "Is the moon made of cheese?", "Do cats bark?", "Is Paris in France?", "Is fire cold?",
              "Do fish swim?", "Is snow hot?", "Can cars fly?", "Is the sun a star?", "Do trees walk?", "Is ice frozen water?",
              "Is two bigger than three?", "Do dogs have tails?", "Is glass transparent?", "Can rocks talk?", "Is milk white?",
              "Do clocks measure time?", "Is the ocean dry?", "Can people breathe underwater?", "Is grass green?", "Do bicycles have wings?",
              "Is a whale a fish?", "Do books have pages?"]
YN_TEST = ["Is it going to rain tomorrow?", "Would you like some tea?", "Is this a good idea?", "Should I take the job?", "Did she call back?",
           "Is the store open now?", "Was the movie any good?", "Do you think he will come?", "Is the answer forty-two?", "Are we there yet?",
           "Will the project finish on time?", "Is the soup ready?", "Does this look right to you?", "Is it too late to apply?", "Can we leave early?",
           "Is the train usually late?", "Did the plan work?", "Is that the last one?", "Was it worth it?", "Do they sell tickets here?"]

CONCEPTS = {
  "tense (past vs present)": dict(
      extract=[t.format(n=s) for s in SUBJ for t in ("{n}", "And then {n}", "As usual, {n}")],
      test=[t.format(n=s) for s in SUBJ for t in ("In the story, {n}", "According to my friend, {n}", "Everyone knows that {n}", "It turns out {n}")],
      pos=["walked", "played", "jumped", "looked", "wanted", "stayed", "moved", "called", "asked", "liked", "worked", "opened", "started", "needed", "lived"],
      neg=["walks", "plays", "jumps", "looks", "wants", "stays", "moves", "calls", "asks", "likes", "works", "opens", "starts", "needs", "lives"],
      sep=" ", gen="In the story, the old man"),
  "number (plural vs singular)": dict(
      extract=[t.format(c=c) for c in CTX for t in ("{c}, I saw the", "{c}, we looked at the", "{c}, she pointed at the")],
      test=[t.format(c=c) for c in CTX for t in ("{c}, they noticed the", "{c}, he photographed the", "{c}, everyone stared at the", "{c}, I remember the")],
      pos=["cats", "dogs", "cars", "books", "trees", "houses", "birds", "boys", "girls", "balls", "horses", "flowers", "boxes", "chairs", "cups"],
      neg=["cat", "dog", "car", "book", "tree", "house", "bird", "boy", "girl", "ball", "horse", "flower", "box", "chair", "cup"],
      sep=" ", gen="At the market, they noticed the"),
  "category (animal vs vehicle)": dict(
      extract=[t.format(c=c) for c in CTX for t in ("{c}, I saw a", "{c}, we looked at a", "{c}, she pointed at a")],
      test=[t.format(c=c) for c in CTX for t in ("{c}, they noticed a", "{c}, he photographed a", "{c}, everyone stared at a", "{c}, I remember a")],
      pos=["dog", "cat", "horse", "bird", "cow", "rabbit", "fox", "deer", "sheep", "goat", "wolf", "bear", "duck", "pig", "mouse"],
      neg=["truck", "car", "bus", "train", "bike", "taxi", "tractor", "van", "boat", "plane", "jeep", "scooter", "ship", "helicopter", "motorcycle"],
      sep=" ", gen="On the road, they noticed a"),
  "answer bias (yes vs no)": dict(
      extract=[f"Question: {q}\nAnswer:" for q in YN_EXTRACT], test=[f"Question: {q}\nAnswer:" for q in YN_TEST],
      pos=["Yes"], neg=["No"], sep=" ", gen="Question: Should we go ahead with the plan?\nAnswer:"),
  "digit magnitude (large vs small)": dict(
      extract=[t.format(c=c) for c in CTX for t in ("{c}, the answer is ", "{c}, the count was ", "{c}, the number is ")],
      test=[t.format(c=c) for c in CTX for t in ("{c}, the total comes to ", "{c}, I would guess ", "{c}, the score was ", "{c}, the result is ")],
      pos=["6", "7", "8", "9"], neg=["1", "2", "3", "4"], sep="", gen="At school, the score was "),
}
IDX = CAP_LAYERS[12]; ALPHAS2 = [-0.4, -0.2, -0.1, 0.1, 0.2, 0.4]
INJECTS = [("L1", [0]), ("L2", [1]), ("L3", [2]), ("L4", [3]), ("all", [0, 1, 2, 3])]
W = model.lm_head.weight.detach().float()

def run_concept(name, C):
    sep = C["sep"]
    ok = lambda w: len(tokenizer.encode(sep + w, add_special_tokens=False)) == 1
    pos, neg = [w for w in C["pos"] if ok(w)], [w for w in C["neg"] if ok(w)]
    pos_ids = [tokenizer.encode(sep + w, add_special_tokens=False)[0] for w in pos]
    neg_ids = [tokenizer.encode(sep + w, add_special_tokens=False)[0] for w in neg]
    npair = min(len(pos), len(neg), 8)
    texts, labels = [], []
    for pre in C["extract"]:
        for i in range(npair):
            texts += [pre + sep + pos[i], pre + sep + neg[i]]; labels += [1, 0]
    # extraction at layer-12 output, last token, per loop
    sums = {s: torch.zeros(2, H, device=model.device) for s in range(S)}; nrm = {s: [0.0, 0] for s in range(S)}
    cur = [None]
    def cap(mod, args, kwargs, out):
        h = (out[0] if isinstance(out, tuple) else out)[:, -1].float(); s = int(kwargs["current_ut"])
        for i, lab in enumerate(cur[0]): sums[s][lab] += h[i]
        nrm[s][0] += h.norm(dim=-1).sum().item(); nrm[s][1] += len(h)
    hc = layers[IDX].register_forward_hook(cap, with_kwargs=True)
    try:
        with torch.no_grad():
            for i in range(0, len(texts), 32):
                enc = tokenizer(texts[i:i+32], return_tensors="pt", padding=True).to(model.device); cur[0] = labels[i:i+32]
                model(**enc, use_cache=False)
    finally: hc.remove()
    n1 = sum(labels); n0 = len(labels) - n1
    v = {s: sums[s][1] / n1 - sums[s][0] / n0 for s in range(S)}
    u = {s: v[s] / v[s].norm() for s in range(S)}; rn = {s: nrm[s][0] / nrm[s][1] for s in range(S)}
    g = torch.Generator().manual_seed(1); ru = {s: (lambda r: r / r.norm())(torch.randn(H, generator=g)).to(model.device) for s in range(S)}
    w_read = W[pos_ids].mean(0) - W[neg_ids].mean(0); w_read = (w_read / w_read.norm()).cpu()
    cosL = torch.nn.functional.cosine_similarity(v[0], v[3], dim=0).item()

    @torch.no_grad()
    def metric():
        ps = []
        for i in range(0, len(C["test"]), 16):
            enc = tokenizer(C["test"][i:i+16], return_tensors="pt", padding=True).to(model.device)
            p = torch.softmax(model(**enc, use_cache=False).logits[:, -1].float(), -1)
            pp, pn = p[:, pos_ids].sum(1), p[:, neg_ids].sum(1); ps += (pp / (pp + pn)).tolist()
        return float(np.mean(ps))
    hs = [layers[IDX].register_forward_hook(make_steer_hook(IDX), with_kwargs=True)]
    out = {"concept": name, "pos": pos, "neg": neg, "n_pairs": n1, "n_test": len(C["test"]), "cos_L1_L4": cosL,
           "vec_norm": {f"L{s+1}": v[s].norm().item() for s in range(S)}, "resid_norm": {f"L{s+1}": rn[s] for s in range(S)}, "sweep": []}
    try:
        steer_cfg.clear(); out["baseline_p"] = metric(); out["baseline_nll"] = neutral_nll()
        for ln, loops in INJECTS:
            for a in ALPHAS2:
                for kind, U in (("caa", u), ("random", ru)):
                    steer_cfg.clear(); steer_cfg[IDX] = {s: a * rn[s] * U[s] for s in loops}
                    out["sweep"].append({"inject": ln, "alpha": a, "kind": kind, "p": metric(), "nll": neutral_nll()})
        # decay tracking: inject at loop 1, readout component at end of each loop (last token) and |delta| at layer-12 output
        capd = {}; cnt = [0]
        def cmid(mod, args, kwargs, o): capd[("mid", int(kwargs["current_ut"]))] = (o[0] if isinstance(o, tuple) else o)[:, -1].detach().float().cpu()
        def cend(mod, i, o): capd[("end", cnt[0])] = o[:, -1].detach().float().cpu(); cnt[0] += 1
        hd = [layers[IDX].register_forward_hook(cmid, with_kwargs=True), model.model.norm.register_forward_hook(cend)]
        def states(cfg):
            steer_cfg.clear(); steer_cfg.update(cfg); acc = {}
            with torch.no_grad():
                for i in range(0, len(C["test"]), 16):
                    enc = tokenizer(C["test"][i:i+16], return_tensors="pt", padding=True).to(model.device); capd.clear(); cnt[0] = 0
                    model(**enc, use_cache=False)
                    for k2, h in capd.items(): acc.setdefault(k2, []).append(h)
            return {k2: torch.cat(vv) for k2, vv in acc.items()}
        try:
            b = states({}); dec = {"along_readout_end": [], "norm_mid": [], "along_caa_mid": []}
            for sign in (1, -1):
                vi = sign * 0.2 * rn[0] * u[0]; st = states({IDX: {0: vi}}); nv = vi.norm().item()
                dec["along_readout_end"].append([sign * ((st[("end", s)] - b[("end", s)]) @ w_read).mean().item() / nv for s in range(S)])
                dec["norm_mid"].append([(st[("mid", s)] - b[("mid", s)]).norm(dim=1).mean().item() / nv for s in range(S)])
                dec["along_caa_mid"].append([sign * ((st[("mid", s)] - b[("mid", s)]) @ u[s].cpu()).mean().item() / nv for s in range(S)])
            out["decay"] = {k2: np.mean(vv, 0).tolist() for k2, vv in dec.items()}
        finally:
            for h in hd: h.remove()
        # generation sample at all loops, alpha +-0.2
        enc = tokenizer(C["gen"], return_tensors="pt").to(model.device); out["gen"] = {}
        for a in (-0.2, 0, 0.2):
            steer_cfg.clear()
            if a: steer_cfg[IDX] = {s: a * rn[s] * u[s] for s in range(S)}
            with torch.no_grad(): o = model.generate(**enc, max_new_tokens=25, do_sample=False, pad_token_id=tokenizer.pad_token_id)
            out["gen"][str(a)] = tokenizer.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    finally:
        for h in hs: h.remove()
        steer_cfg.clear()
    return out

def get(o, inj, a, kind): return next(r for r in o["sweep"] if r["inject"] == inj and r["alpha"] == a and r["kind"] == kind)
all_out = []; t0 = time.time()
for name, C in CONCEPTS.items():
    o = run_concept(name, C); all_out.append(o)
    print(f"\n=== {name}  ({len(o['pos'])}+{len(o['neg'])} words, {o['n_pairs']} pairs, {o['n_test']} test prompts; baseline p={o['baseline_p']:.3f}, "
          f"cos(v_L1, v_L4)={o['cos_L1_L4']:.2f})   [{time.time()-t0:.0f}s]")
    print(f"{'inject':<7}{'a=-0.2 caa[rand]':>20}{'a=+0.2 caa[rand]':>20}{'specific':>10}{'nll@+0.2':>10}{'a=+-0.4 caa':>16}")
    for ln, _ in INJECTS:
        cm, cp = get(o, ln, -0.2, "caa"), get(o, ln, 0.2, "caa"); rm, rp = get(o, ln, -0.2, "random"), get(o, ln, 0.2, "random")
        spec = (cp["p"] - cm["p"]) - (rp["p"] - rm["p"])
        print(f"{ln:<7}{cm['p']:>12.3f} [{rm['p']:.2f}]{cp['p']:>12.3f} [{rp['p']:.2f}]{spec:>10.3f}{cp['nll']:>10.2f}"
              f"{get(o, ln, -0.4, 'caa')['p']:>9.3f}/{get(o, ln, 0.4, 'caa')['p']:.3f}")
    d = o["decay"]
    print(f"decay from loop-1 injection: readout component {' -> '.join(f'{x:+.2f}' for x in d['along_readout_end'])} | "
          f"|delta| at layer 12 {' -> '.join(f'{x:.2f}' for x in d['norm_mid'])} | along CAA dir {' -> '.join(f'{x:+.2f}' for x in d['along_caa_mid'])}")
    for a, txt in o["gen"].items(): print(f"  gen a={float(a):+.1f}: {txt!r}")

print("\n\n=== Cross-concept summary (after 12 layers, |alpha| = 0.2) ===")
print(f"{'concept':<34}{'base':>6}{'spec L1':>9}{'L2':>7}{'L3':>7}{'L4':>7}{'all':>7}{'nll all':>9}{'cos14':>7}{'readout decay/loop':>20}")
for o in all_out:
    specs = []
    for ln, _ in INJECTS:
        cm, cp, rm, rp = get(o, ln, -0.2, "caa"), get(o, ln, 0.2, "caa"), get(o, ln, -0.2, "random"), get(o, ln, 0.2, "random")
        specs.append((cp["p"] - cm["p"]) - (rp["p"] - rm["p"]))
    r = o["decay"]["along_readout_end"]; ratios = [r[i+1] / r[i] for i in range(S - 1) if abs(r[i]) > 1e-6]
    print(f"{o['concept']:<34}{o['baseline_p']:>6.2f}" + "".join(f"{x:>7.2f}" for x in specs).replace(f"{specs[0]:>7.2f}", f"{specs[0]:>9.2f}", 1)
          + f"{get(o, 'all', 0.2, 'caa')['nll']:>9.2f}{o['cos_L1_L4']:>7.2f}{'  ' + ' '.join(f'{x:.2f}' for x in ratios):>20}")
print(f"(baseline neutral-text NLL {all_out[0]['baseline_nll']:.2f}; 'spec' = CAA shift minus random shift between alpha -0.2 and +0.2)")
json.dump(all_out, open(f"{OUT}/caa_concepts.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{OUT}/caa_concepts.json", path_in_repo="caa/caa_concepts.json", repo_id=REPO, repo_type="dataset",
                commit_message="CAA on five more concepts with loop sweep, controls, damage and decay")
print("saved + pushed caa/caa_concepts.json")


=== tense (past vs present)  (15+15 words, 384 pairs, 64 test prompts; baseline p=0.323, cos(v_L1, v_L4)=0.90)   [62s]
inject     a=-0.2 caa[rand]    a=+0.2 caa[rand]  specific  nll@+0.2     a=+-0.4 caa
L1            0.282 [0.33]       0.696 [0.31]     0.431      1.45    0.263/0.767
L2            0.245 [0.32]       0.732 [0.33]     0.475      1.45    0.202/0.838
L3            0.181 [0.33]       0.764 [0.31]     0.603      1.40    0.115/0.876
L4            0.055 [0.35]       0.875 [0.30]     0.871      1.45    0.023/0.970
all           0.023 [0.36]       0.986 [0.28]     1.038      1.99    0.091/0.990
decay from loop-1 injection: readout component +1.30 -> +1.45 -> +1.36 -> +1.18 | |delta| at layer 12 1.00 -> 1.47 -> 1.39 -> 1.18 | along CAA dir +1.00 -> +0.38 -> +0.30 -> +0.23
  gen a=-0.2: ' is the one who is the protagonist, and the other characters are the ones who are the antagonists. The old man is the'
  gen a=+0.0: "'s wife is the one who is the main character, and the old man 